In [1]:
!pip install openai pandas scikit-learn

  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached scipy-1.17.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 8.9 MB/s  0:00:00
Using cached anyio-4.13.0-py3-none-any.whl (114 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 8.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 7.2 MB/s  0:00:01 eta 0:00:01
Using cached h

# 2차 시도

In [10]:
import pandas as pd

# 프롬프트 예시 리뷰번호+sent_id 
# 기존 26개 + 새로 추가 3개
fewshot_keys = [
    (37987297, 0), (18155429, 0), (66929759, 1), (57569307, 3),
    (48037025, 1), (61696159, 2), (76638272, 1), (13673986, 0),
    (21730866, 2), (6446270, 0), (15721995, 1), (850337, 1),
    (8548638, 2), (59687437, 0), (47475156, 0), (31840046, 0),
    (30944195, 1), (32384345, 2), (58226674, 1), (7318447, 0),
    (59396725, 1), (50629439, 9), (45479996, 0), (22312384, 0),
    (45467650, 0), (64634192, 0),
    # 새로 추가
    (45730337, 1), (29786430, 1)
]

df = pd.read_csv('step7_3c_labeled_v10(500).csv', encoding='utf-8-sig')
df['key'] = list(zip(df['리뷰번호'], df['sent_id']))

few_shot = df[df['key'].isin(fewshot_keys)].drop(columns=['key']).reset_index(drop=True)
eval_df = df[~df['key'].isin(fewshot_keys)].drop(columns=['key']).reset_index(drop=True)

few_shot.to_csv('step7_3c_fewshot_v2.csv', index=False, encoding='utf-8-sig')
eval_df.to_csv('step7_3c_eval_v2.csv', index=False, encoding='utf-8-sig')

print(f'few-shot: {len(few_shot)}건')
print(f'평가용: {len(eval_df)}건')
print(eval_df['human_label'].value_counts())

few-shot: 28건
평가용: 472건
human_label
positive    223
negative    126
neutral     123
Name: count, dtype: int64


In [ ]:
# API 키 설정
client = openai.OpenAI(api_key="")

# 프롬프트
system_prompt = """당신은 패션 리뷰 감성 분석 전문가입니다.
아래 기준에 따라 주어진 문장을 positive / negative / neutral 중 하나로 분류하세요.

판단 기준: 반드시 주어진 [문장] 기준으로 판단하세요.
[원문]은 문장의 맥락 파악을 위한 참고 정보입니다. 원문 전체 분위기에 끌려가지 마세요.

---
[분류 기준]

[POSITIVE]
- 명확한 만족/칭찬: "정말 마음에 들어요", "색깔이 너무 예뻐요"
- 가성비/기대 이상: "이 가격에 이 퀄리티면 완전 이득", "저렴한데 생각보다 너무 좋아요"
- 추천/재구매 의사: "고민하지 말고 사세요", "다음에 또 살 것 같아요"
- 반복 사용/애착 표현: "자꾸 이것만 입게 돼요", "매일 입고 싶어요"
- 약한 긍정 (조건부 포함): "허리 조이면 핏 괜찮아요", "세탁하면 더 나을 것 같아요"

[NEGATIVE]
- 명확한 불만/실망: "생각보다 너무 실망스러워요", "재질이 완전 별로예요"
- 후회/대안 표현: "한 사이즈 크게 살걸 그랬어요", "기모로 살걸 후회돼요"
- 걱정/우려: "자꾸 늘어날까봐 걱정돼요", "세탁하면 줄어들까봐 불안해요"
- 반어법/우회 표현: "두 달 만에 왔네요 ㅋㅋ", "택배가 참 여유롭네요" (맥락 파악 필요)
- 배송 불만: "한 달 넘게 기다렸어요", "배송이 너무 느려서 환불하고 싶었어요"
- 약한 부정 (조건부 포함): "크긴 한데 어쩔 수 없죠 뭐", "이 소재 특성상 보풀은 감수해야죠"

[NEUTRAL]
- 단순 사실/정보 전달: "두께는 얇은 편이에요", "허리 조절 끈이 있어요"
- 체형/개인 상황 진술: "저는 키가 작은 편이라 S 샀어요", "살이 좀 쪄서 M으로 올렸어요"
- 구매/착용 경험 서술: "두 번 빨았는데 아직 멀쩡해요", "작년에 사서 지금도 입고 있어요"
- 옷과 무관한 아쉬움: "날씨가 더워져서 아직 못 꺼냈어요", "겨울이 끝나서 아쉬워요"

---
[애매한 케이스 규칙]
- "괜찮아요" → 약한 긍정이면 positive, 미온적이면 neutral
- 긍/부정 혼재 → 마지막 톤 기준. 비중 비슷하면 neutral
- ㅠㅠ/ㅜㅜ 있어도 체형 한탄이면 neutral (옷 평가 아님)
- 걱정/우려가 문장의 결론이면 negative
- "걱정했는데", "걱정했지만" → 원문에서 결과 확인 후 판단. 결과가 좋으면 neutral
- 반어법 → 맥락 파악 후 negative
- 문장 잘림이지만 감성 명확 → 나온 부분 기준으로 판단
- 원문 참고 시 → 해당 문장/aspect와 직접 연결될 때만 반영. 원문 전체 분위기에 끌려가지 않기

---
[예시]
문장: 체크셔츠 찾다가 사게되었는데
원문: 체크셔츠 찾다가 사게되었는데 핏도 괜찮고 좋네요.
→ {"label": "neutral", "reason": "단순 사실/정보 전달"}

문장: 사이즈는 한사이즈업했습니다.
원문: 상품은 봄 여름에 입기좋은 얇기입니다. 사이즈는 한사이즈업했습니다. 저스트입니다.
→ {"label": "neutral", "reason": "단순 사실/정보 전달"}

문장: 촉감도 좋고 만족합니다.
원문: 좋은 가격에 잘 산 것 같습니다. 기본템으로 무난하게 입고 있어요. 촉감도 좋고 만족합니다. 세일할 때 사세요.
→ {"label": "positive", "reason": "명확한 만족/칭찬"}

문장: 운동하거나 집 앞 나갈 때 편하게 입기 좋아요
원문: 운동하거나 집 앞 나갈 때 편하게 입기 좋아요 시원해지면 상의 가을 옷으로 입어도 좋을 것 같아요
→ {"label": "positive", "reason": "명확한 만족/칭찬"}

문장: 이 가격이면 가성비 갑인거같아요.
원문: 사이즈도 적당하고 페인팅도 괜찮아요. 이 가격이면 가성비 갑인거같아요.
→ {"label": "positive", "reason": "가성비/기대 이상"}

문장: 저렴한가격에 좋은재질의 옷을 구매한거같아 기분이 좋습니다.
원문: 안에 기모도 폭신폭신하고 가을 초겨울에 외투없이 입기도 두께가 적당한 것 같네요 저렴한가격에 좋은재질의 옷을 구매한거같아 기분이 좋습니다.
→ {"label": "positive", "reason": "가성비/기대 이상"}

문장: 물빠질까봐 두렵지만
원문: 물빠질까봐 두렵지만 색상 예쁘고 넉넉한폼에 만족합니다
→ {"label": "negative", "reason": "걱정/우려"}

문장: 입고 벗기에는 조금 불편하고 늘어날까봐 걱정돼요
원문: 프린팅도 예쁘고 기장도 맘에 들어요 근데 목부분이 좁아서? 입고 벗기에는 조금 불편하고 늘어날까봐 걱정돼요
→ {"label": "negative", "reason": "걱정/우려"}

문장: 살이 너무 많이 쪄버리는 바람에 맞는 옷 구매하기도 어렵고 체형 보정이 되는 옷이 필요했어요.
원문: 살이 너무 많이 쪄버리는 바람에 맞는 옷 구매하기도 어렵고 체형 보정이 되는 옷이 필요했어요. 적당히 두툼하면서 부드러운 재질이라 보정도 잘되고, 오버핏이라 기장이 살짝 길지만 옆트임이 잇어서 불편하지 않고 이쁘네요.
→ {"label": "neutral", "reason": "체형/개인 상황 진술"}

문장: 배나온 체형에 다리는 날씬한 편이구요.
원문: 배나온 체형에 다리는 날씬한 편이구요. 사이즈 딱 예쁘게 맞아요
→ {"label": "neutral", "reason": "체형/개인 상황 진술"}

문장: 생각보다 저랑 너무 안어울려요 ㅠㅠ
원문: 생각보다 저랑 너무 안어울려요 ㅠㅠ 이런 펑키..? 스타일(?)이 안맞는 것같아오 ㅜ 넘 아쉽습니다  핏도 별로라서 일단 건조기 돌려볼게용 ㅠ
→ {"label": "negative", "reason": "명확한 불만/실망"}

문장: 실 다 붙어있고 늘어져있고 거기서도 실망했어요.
원문: 💟배송 - 배송지연이라길래 많이 늦을 줄 알았는데 이틀만에 도착했어요! 좀 급하게 산거라서 많이 걱정했는데 빨라서 매우 만조쿠🩷🩷  💟사이즈 - 키가 158이라서 S를 구매했는데 엉덩이를 다 덮지도 않고 그렇다고 안 덮지도 않고 딱 적당한 길이여서 사이즈도 만족!! 150중후반이신 분들은 S 사시길🫶🏻  🔼상품 - 옷을 보았는데 이상한 냄새가 났어요 (너무 가깝게 맡은 것도 아님!!) 그 부분에서 좀 실망했고 뒤에 목부분에 마무리가 전혀 깔끔하게 마감이 안 되어있어요 실 다 붙어있고 늘어져있고 거기서도 실망했어요. 하지만 티는 안 남! 그리고 냅다 실밥들이 풀어져서 뭉쳐있길래 불량인가 했어요.. - 꾸안꾸하기 조흠🩷
→ {"label": "negative", "reason": "명확한 불만/실망"}

문장: 배송이 늦어서 주문한 걸 잊었을 때 쯤 왔지만
원문: 배송이 늦어서 주문한 걸 잊었을 때 쯤 왔지만 사이즈도 딱 맞고 실물이 이뻐요~ 사진을 너무 대충 찍었지만 실물은 이쁩니다 ㅋㅋ 거울 좀 닦아야겠네..
→ {"label": "negative", "reason": "배송 불만"}

문장: 배송 느린 거?
원문: 개예쁨~~배송 느린 거? 참을 수 있다~이 말입니다 고민고민하지 말고 사세욤
→ {"label": "negative", "reason": "배송 불만"}

문장: 되게 무난한 상의를 좋은 가격에 구매한거 같아서 기분이 좋습니다.
원문: 되게 무난한 상의를 좋은 가격에 구매한거 같아서 기분이 좋습니다.
→ {"label": "positive", "reason": "약한 긍정 (조건부 포함)"}

문장: 스트링 조이면 핏 괜찮아요
원문: 은근 얇긴 해요 예상했지만 역시나 크긴합니다 스트링 조이면 핏 괜찮아요 다만 기분탓인지 스트링이 꽉 고정되는 느낌은 아니에요..
→ {"label": "positive", "reason": "약한 긍정 (조건부 포함)"}

문장: 도 크네요
원문: xs사이즈인데도 크네요 수선맡겨서 입을 생각입니다 바지자체는 예쁜데 가랑이 모양이 좀 안잡혀요
→ {"label": "negative", "reason": "약한 부정 (조건부 포함)"}

문장: 일단 어떻게 코디해야할지 막막한데..
원문: 일단 어떻게 코디해야할지 막막한데.. 저랑 잘 어울리는지도 잘 모르겠네요 그래도 일단 만족해서 예쁘게 코디해서 입어보려고 합니당ㅎㅎ
→ {"label": "negative", "reason": "약한 부정 (조건부 포함)"}

문장: 추천입니다!
원문: 이거 가을부터 입으면 간지날것 같습니다 너무 맘에들어요 추천입니다!
→ {"label": "positive", "reason": "추천/재구매 의사"}

문장: 이뻐 잘어울려어디든 디자인이너무이뻐 굿 추천
원문: 이뻐이뻐 잘어울려어디든 디자인이너무이뻐 굿 추천
→ {"label": "positive", "reason": "추천/재구매 의사"}

문장: 추워 돌아가시는 날씨에도 자꾸 이것만 입을라구 해서 걱정이네요
원문: 친구 사주고 좋아해서 엄마도 사줬는데 추워 돌아가시는 날씨에도 자꾸 이것만 입을라구 해서 걱정이네요
→ {"label": "positive", "reason": "반복 사용/애착 표현"}

문장: 옷이 딱 맞고 색감이 이뻐서 어디에나 어울릴 거 같아서 좋았고 맨날 입고 다닐만큼 이뻐요
원문: 옷이 딱 맞고 색감이 이뻐서 어디에나 어울릴 거 같아서 좋았고 맨날 입고 다닐만큼 이뻐요
→ {"label": "positive", "reason": "반복 사용/애착 표현"}

문장: 세 번째 빨래 기준 막 심하지는 않습니다.
원문: 물빠짐이 심하다고 했지만 세 번째 빨래 기준 막 심하지는 않습니다.
→ {"label": "neutral", "reason": "구매/착용 경험 서술"}

문장: 사이즈 m살걸 그랬어요
원문: 사이즈 m살걸 그랬어요 박시핏으로 입으려 했는데 정닥하게 맞는 정도네요 실밥 처리가 잘 안돼 있어서 아쉬워요
→ {"label": "negative", "reason": "후회/대안 표현"}

문장: 색감도 괜찮고 라지가 생각보다 커서 미듐살껄 그랬어요
원문: 색감도 괜찮고 라지가 생각보다 커서 미듐살껄 그랬어요 재질이 좀 입다보면 보플 일어나기 쉬운 재질이네요 그점이 조금 아쉬워요
→ {"label": "negative", "reason": "후회/대안 표현"}

문장: 여름이 지나가서 아쉬운 마음이에요
원문: 너무 마음에 드는 옷 입니다 계속 입고 싶은데 여름이 지나가서 아쉬운 마음이에요
→ {"label": "neutral", "reason": "옷과 무관한 아쉬움"}

# 1차 시도 때 llm이 틀린 유형 프롬프트에 추가 
문장: 오버핏이라 많이 클까봐 걱정했는데
원문: 오버핏이라 많이 클까봐 걱정했는데 딱 적당한 오버핏이라 너무 좋아요
→ {"label": "neutral", "reason": "걱정했지만 결과가 좋아 neutral"}

문장: 여름에 잘입을 것 같습니다
원문: 사진대로 이쁘네요 여름에 잘입을 것 같습니다 많이 파세용
→ {"label": "neutral", "reason": "계절 착용 추측, 옷 직접 평가 아님"}

---

반드시 JSON으로만 응답하세요. 다른 텍스트 없이:
{"label": "positive/negative/neutral", "confidence": 0.0~1.0, "reason": "한줄 이유"}"""

# CSV 불러오기
df = pd.read_csv('step7_3c_eval_v2.csv', encoding='utf-8-sig')

def label_sentence(row):
    user_content = f"[문장]: {row['sentence']}\n[원문]: {row['리뷰내용']}"
    
    try:
        response = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=0
        )
        text = response.choices[0].message.content.strip()
        text = text.replace('```json', '').replace('```', '').strip()
        result = json.loads(text)
        return result['label'], result['confidence'], result['reason']
    except Exception as e:
        print(f"에러: {e} | 문장: {row['sentence']}")
        return None, None, None

# 실행
results = []
for i, row in df.iterrows():
    label, conf, reason = label_sentence(row)
    results.append({
        'llm_label': label,
        'llm_confidence': conf,
        'llm_reason': reason
    })
    if i % 50 == 0:
        print(f'{i}/{len(df)} 완료...')
    time.sleep(0.1)

# 결과 저장
df_result = pd.concat([df, pd.DataFrame(results)], axis=1)
df_result.to_csv('step7_3c_llm_pilot_v3.csv', index=False, encoding='utf-8-sig')
print('완료!')

# 평가
from sklearn.metrics import classification_report
eval_df = df_result.dropna(subset=['llm_label'])
print(classification_report(eval_df['human_label'], eval_df['llm_label'], digits=4))
print(pd.crosstab(eval_df['human_label'], eval_df['llm_label'],
                  rownames=['실제(사람)'], colnames=['예측(LLM)']))

0/472 완료...
50/472 완료...
100/472 완료...
150/472 완료...
200/472 완료...
250/472 완료...
300/472 완료...
350/472 완료...
400/472 완료...
450/472 완료...
완료!
              precision    recall  f1-score   support

    negative     0.9280    0.9206    0.9243       126
     neutral     0.7445    0.8293    0.7846       123
    positive     0.9381    0.8834    0.9099       223

    accuracy                         0.8792       472
   macro avg     0.8702    0.8778    0.8729       472
weighted avg     0.8850    0.8792    0.8811       472

예측(LLM)   negative  neutral  positive
실제(사람)                               
negative       116        9         1
neutral          9      102        12
positive         0       26       197


In [ ]:
df_result = pd.read_csv('step7_3c_llm_pilot_v3.csv', encoding='utf-8-sig')

# 틀린 것만 필터
wrong = df_result[df_result['human_label'] != df_result['llm_label']]

# 보기 좋게 출력
print(f'총 오분류: {len(wrong)}건\n')
print(wrong[['sentence', 'human_label', 'llm_label', 'llm_reason']].to_string())

In [13]:
import pandas as pd

df = pd.read_csv('step7_3c_llm_pilot_v3.csv', encoding='utf-8-sig')

cases = [
    ('neutral', 'positive'),
    ('neutral', 'negative'),
    ('positive', 'neutral'),
    ('positive', 'negative'),
    ('negative', 'neutral'),
    ('negative', 'positive'),
]

for human, llm in cases:
    wrong = df[(df['human_label']==human) & (df['llm_label']==llm)].reset_index(drop=True)
    if len(wrong) > 0:
        fname = f'wrong_{human}_to_{llm}.csv'
        wrong.to_csv(fname, index=False, encoding='utf-8-sig')
        print(f'{human} → {llm}: {len(wrong)}건 → {fname}')

neutral → positive: 12건 → wrong_neutral_to_positive.csv
neutral → negative: 9건 → wrong_neutral_to_negative.csv
positive → neutral: 26건 → wrong_positive_to_neutral.csv
negative → neutral: 9건 → wrong_negative_to_neutral.csv
negative → positive: 1건 → wrong_negative_to_positive.csv
